## algorithm design and anlysis-2026 spring  homework 2
**Deadline**：2026.5.20

**name**:


note：
---
1. 本题目为在线OJ作业，OJ平台题目链接：https://www.nowcoder.com/acm/contest/129481，访问密码见课件；
3. 在OJ平台运行通过后，将源码复制到本文件对应题目的下方代码框中；
4. 如若作答有雷同，全部取消成绩；


## A 排序

In [ ]:
## add your code here
import sys

sys.setrecursionlimit(1000000)


class Builder:
    def __init__(self, arr):
        self.a = arr
        self.n = len(arr)
        self.ops = []

    def build(self):
        n = self.n
        seen = [0] * n

        for x in self.a:
            if x < 0 or x >= n:
                return False
            seen[x] = 1

        if sum(seen) != n:
            return False

        if n == 1:
            return True

        half = n // 2

        left = Builder([self.a[i * 2] // 2 for i in range(half)])
        right = Builder([self.a[i * 2 + 1] // 2 for i in range(half)])

        if not left.build() or not right.build():
            return False

        ops = []

        if self.a[0] & 1:
            ops.append(1 if n == 2 else -1)

        xl = 0
        for x in left.ops:
            if x > 0:
                ops.append(-1)
                ops.append(1)
            else:
                y = (-x) * 2
                ops.append(-y)
                xl ^= y

        if xl:
            ops.append(-xl)

        xr = 0
        for x in right.ops:
            if x > 0:
                ops.append(1)
                ops.append(-1)
            else:
                y = (-x) * 2
                ops.append(-y)
                xr ^= y

        if (xl & half) != (xr & half):
            return False

        xl %= half
        xr %= half

        if xl != xr:
            return False

        merged = []
        for x in ops:
            if merged and x < 0 and merged[-1] < 0:
                merged[-1] = -((-merged[-1]) ^ (-x))
                if merged[-1] == 0:
                    merged.pop()
            else:
                merged.append(x)

        self.ops = merged
        return True


def solve():
    data = list(map(int, sys.stdin.buffer.read().split()))
    if not data:
        return

    n = data[0]
    A = data[1]
    B = data[2]
    p = data[3:3 + n]

    ans = []

    diff = (A - B) % n
    unit = diff & -diff
    if unit == 0:
        unit = n

    def add_magic(v):
        nonlocal p

        v %= n
        if v == 0:
            return

        ans.append((2, v))
        p = [(x + v) % n for x in p]

    def xor_magic(v):
        nonlocal p

        if v == 0:
            return

        ans.append((1, v))
        p = [x ^ v for x in p]

    def swap_magic():
        nonlocal p

        ans.append((0, 0))

        a, b = A, B
        p = [b if x == a else a if x == b else x for x in p]

    def pair_pos(x, y):
        delta = (y - x + n - unit + n) % n

        px = 0
        py = 0

        step = n >> 1
        while step >= 2 * unit:
            if delta >= step:
                delta -= step
                py += step >> 1
            else:
                px += step >> 1
            step >>= 1

        low = x & (unit - 1)

        px += (n >> 1) + low
        py += low

        return px, py

    def do_swap(x, y):
        bx = (x // unit) & 1
        by = (y // unit) & 1

        if bx == by:
            if bx == 0:
                mid = (x & (unit - 1)) + unit
            else:
                mid = x & (unit - 1)

            do_swap(x, mid)
            do_swap(y, mid)
            do_swap(x, mid)
            return

        pa, pb = pair_pos(A, B)
        px, py = pair_pos(x, y)

        add_magic(px - x)
        xor_magic(px ^ pa)
        add_magic(A - pa)

        swap_magic()

        add_magic(pa - A)
        xor_magic(px ^ pa)
        add_magic(x - px)

    if unit > 1:
        base = [p[i] & (unit - 1) for i in range(unit)]
        builder = Builder(base)

        if not builder.build():
            print(-1)
            return

        for x in builder.ops:
            if x > 0:
                add_magic(x)
            else:
                xor_magic(-x)

    for rem in range(unit):
        cur = p[rem:n:unit]
        cur.sort()

        need = list(range(rem, n, unit))
        if cur != need:
            print(-1)
            return

        for j in range(rem, n, unit):
            if p[j] != j:
                do_swap(j, p[j])

                if len(ans) > 32768:
                    print(-1)
                    return

    for i in range(n):
        if p[i] != i:
            print(-1)
            return

    if len(ans) > 32768:
        print(-1)
        return

    out = [str(len(ans))]

    for t, v in ans:
        if t == 0:
            out.append("0")
        else:
            out.append(f"{t} {v}")

    sys.stdout.write("\n".join(out))


if __name__ == "__main__":
    solve()

## B 长跑

In [ ]:
## add your code here
import sys
import heapq


def can_reach(n, L, maxn, S, stations):
    # 起点直接到终点
    if L <= maxn:
        return True

    # 同一个位置可能有多个补给点，取最便宜的
    best = {}
    for p, c in stations:
        if 0 < p < L:
            if p not in best or c < best[p]:
                best[p] = c

    arr = sorted(best.items())

    # 堆中元素：(当前最小花费, 位置)
    # 起点 0：体力满，花费 0
    heap = [(0, 0)]

    for p, c in arr:
        # 删除无法到达当前位置的状态
        while heap and heap[0][1] + maxn < p:
            heapq.heappop(heap)

        if not heap:
            continue

        cost = heap[0][0] + c

        if cost <= S:
            heapq.heappush(heap, (cost, p))

    # 判断能否从某个已经补满体力的位置跑到终点
    while heap and heap[0][1] + maxn < L:
        heapq.heappop(heap)

    return bool(heap and heap[0][0] <= S)


def main():
    data = list(map(int, sys.stdin.buffer.read().split()))
    idx = 0
    out = []

    while idx < len(data):
        if idx + 4 > len(data):
            break

        n = data[idx]
        L = data[idx + 1]
        maxn = data[idx + 2]
        S = data[idx + 3]
        idx += 4

        stations = []
        for _ in range(n):
            p = data[idx]
            c = data[idx + 1]
            idx += 2
            stations.append((p, c))

        out.append("Yes" if can_reach(n, L, maxn, S, stations) else "No")

    sys.stdout.write("\n".join(out))


if __name__ == "__main__":
    main()

## C 最长回文

In [ ]:
## add your code here
import sys

def solve():
    # 快速读取所有输入，避免 I/O 超时
    input_data = sys.stdin.read().split()
    if not input_data:
        return
    n = int(input_data[0])
    A = input_data[1]
    B = input_data[2]

    # Manacher 算法，返回原生长数组 P
    def manacher(s):
        T = ['^', '#']
        for char in s:
            T.append(char)
            T.append('#')
        T.append('$')
        m = len(T)
        P = [0] * m
        C = 0
        R = 0
        for i in range(1, m - 1):
            i_mirror = 2 * C - i
            if R > i:
                P[i] = min(R - i, P[i_mirror])
            else:
                P[i] = 0
            while T[i + 1 + P[i]] == T[i - 1 - P[i]]:
                P[i] += 1
            if i + P[i] > R:
                C = i
                R = i + P[i]
        return P

    PA = manacher(A)
    PB = manacher(B)

    # 字符串哈希预处理 (内联展开加速运算)
    MOD = (1 << 61) - 1   # 使用大质数(Mersenne prime)避免碰撞
    BASE = 313

    pw = [1] * (n + 1)
    for i in range(1, n + 1):
        pw[i] = (pw[i - 1] * BASE) % MOD

    hash_B = [0] * (n + 1)
    for i in range(n):
        hash_B[i + 1] = (hash_B[i] * BASE + ord(B[i])) % MOD

    A_R = A[::-1]
    hash_AR = [0] * (n + 1)
    for i in range(n):
        hash_AR[i + 1] = (hash_AR[i] * BASE + ord(A_R[i])) % MOD

    max_len = 0
    
    # 本地变量化提升 Python 字典寻址速度
    hAR = hash_AR
    hB = hash_B
    p = pw

    # 遍历情况 1：回文中心落在字符串 A 的子串上
    for j in range(2 * n + 1):
        L_p = PA[j + 1]
        u = (j - L_p) // 2
        v = (j + L_p) // 2

        # u 和 v 分别为原串A中回文串的左右边界(左闭右开)
        st_AR = n - u
        st_B = v - 1
        
        if st_B >= 0 and st_AR >= 0:
            max_K = n - st_AR if st_AR > st_B else n - st_B
            if max_K < 0: max_K = 0
            
            # O(1) 剪枝：只有预期最大长度比现有的好，才进行二分检查
            K_needed = (max_len - L_p) // 2 + 1
            if K_needed <= max_K:
                # 检查达成目标增量的关键哈希，如果不相等直接跳过以极速过滤掉 99% 的错误答案
                if K_needed <= 0 or (hAR[st_AR + K_needed] - hAR[st_AR] * p[K_needed]) % MOD == (hB[st_B + K_needed] - hB[st_B] * p[K_needed]) % MOD:
                    low = K_needed if K_needed > 0 else 1
                    high = max_K
                    ans = low - 1
                    while low <= high:
                        mid = (low + high) >> 1
                        if (hAR[st_AR + mid] - hAR[st_AR] * p[mid]) % MOD == (hB[st_B + mid] - hB[st_B] * p[mid]) % MOD:
                            ans = mid
                            low = mid + 1
                        else:
                            high = mid - 1
                    if L_p + 2 * ans > max_len:
                        max_len = L_p + 2 * ans
        else:
            if L_p > max_len:
                max_len = L_p

    # 遍历情况 2：回文中心落在字符串 B 的子串上
    for j in range(2 * n + 1):
        L_p = PB[j + 1]
        u = (j - L_p) // 2
        v = (j + L_p) // 2

        st_AR = n - 1 - u
        st_B = v
        if st_AR >= 0 and st_B >= 0:
            max_K = n - st_AR if st_AR > st_B else n - st_B
            if max_K < 0: max_K = 0
            
            K_needed = (max_len - L_p) // 2 + 1
            if K_needed <= max_K:
                if K_needed <= 0 or (hAR[st_AR + K_needed] - hAR[st_AR] * p[K_needed]) % MOD == (hB[st_B + K_needed] - hB[st_B] * p[K_needed]) % MOD:
                    low = K_needed if K_needed > 0 else 1
                    high = max_K
                    ans = low - 1
                    while low <= high:
                        mid = (low + high) >> 1
                        if (hAR[st_AR + mid] - hAR[st_AR] * p[mid]) % MOD == (hB[st_B + mid] - hB[st_B] * p[mid]) % MOD:
                            ans = mid
                            low = mid + 1
                        else:
                            high = mid - 1
                    if L_p + 2 * ans > max_len:
                        max_len = L_p + 2 * ans
        else:
            if L_p > max_len:
                max_len = L_p

    print(max_len)

if __name__ == '__main__':
    solve()

## D 优惠券

In [ ]:
#本题使用的C++代码解答，所有在单元格中会有报错提醒
## add your code here
#include <iostream>
#include <cstring>
#include <set>
using namespace std;

// 最大数据上限 与原代码保持一致
const int MAX_SIZE = 100010;

// 物品状态数组：0=无/已出库，1=在库
int item_state[MAX_SIZE];
// 记录每个物品最后一次操作的位置
int last_oper[MAX_SIZE];

int main() {
    // 加速cin输入，效率等同scanf
    ios::sync_with_stdio(false);
    cin.tie(0);

    int op_count;
    // 多组测试用例，读到文件结束
    while (cin >> op_count) {
        // 初始化数组清零
        memset(item_state, 0, sizeof(item_state));
        memset(last_oper, 0, sizeof(last_oper));

        // set模拟原数组平衡树，存储?的位置
        set<int> st;
        // 是否出错标记
        bool has_error = false;
        // 第一个错误的位置，默认-1
        int err_pos = -1;

        for (int i = 1; i <= op_count; ++i) {
            char op_type;
            cin >> op_type;

            // ==============================================
            // 核心规则1：无论是否出错，? 都直接插入集合
            // ==============================================
            if (op_type == '?') {
                if (!has_error) {
                    st.insert(i);
                }
                continue;
            }

            // ==============================================
            // 核心规则2：无论是否出错，必须读取物品编号
            // ==============================================
            int item_id;
            cin >> item_id;

            // 已经出错，只需要更新最后操作位置，直接跳过
            if (has_error) {
                last_oper[item_id] = i;
                continue;
            }

            // ==============================================
            // 入库操作 I
            // ==============================================
            if (op_type == 'I') {
                if (item_state[item_id] == 0) {
                    // 正常入库
                    item_state[item_id] = 1;
                } else {
                    // 重复入库，需要消耗一个?
                    auto it = st.lower_bound(last_oper[item_id]);
                    if (it != st.end()) {
                        st.erase(it);
                    } else {
                        // 无?可消耗，标记错误
                        has_error = true;
                        err_pos = i;
                    }
                }
            }

            // ==============================================
            // 出库操作 O
            // ==============================================
            else if (op_type == 'O') {
                if (item_state[item_id] == 1) {
                    // 正常出库
                    item_state[item_id] = 0;
                } else {
                    // 非法出库，需要消耗一个?
                    auto it = st.lower_bound(last_oper[item_id]);
                    if (it != st.end()) {
                        st.erase(it);
                    } else {
                        // 无?可消耗，标记错误
                        has_error = true;
                        err_pos = i;
                    }
                }
            }

            // ==============================================
            // 核心规则3：无论是否合法，必须更新最后操作位置
            // ==============================================
            last_oper[item_id] = i;
        }

        // 输出第一个错误位置
        cout << err_pos << '\n';
    }
    return 0;
}

## E 任意点

In [ ]:
## add your code here
import sys

class DSU:
    def __init__(self, n):
        self.parent = list(range(n))
        self.rank = [0] * n

    def find(self, x):
        while self.parent[x] != x:
            self.parent[x] = self.parent[self.parent[x]]
            x = self.parent[x]
        return x

    def union(self, a, b):
        ra = self.find(a)
        rb = self.find(b)

        if ra == rb:
            return

        if self.rank[ra] < self.rank[rb]:
            ra, rb = rb, ra

        self.parent[rb] = ra

        if self.rank[ra] == self.rank[rb]:
            self.rank[ra] += 1


def main():
    data = list(map(int, sys.stdin.buffer.read().split()))
    if not data:
        return

    n = data[0]
    dsu = DSU(n)

    x_map = {}
    y_map = {}

    idx = 1
    for i in range(n):
        x = data[idx]
        y = data[idx + 1]
        idx += 2

        if x in x_map:
            dsu.union(i, x_map[x])
        else:
            x_map[x] = i

        if y in y_map:
            dsu.union(i, y_map[y])
        else:
            y_map[y] = i

    components = set()
    for i in range(n):
        components.add(dsu.find(i))

    print(len(components) - 1)


if __name__ == "__main__":
    main()

## F 通配符匹配

In [ ]:
#本题使用的C++代码解答，所有在单元格中会有报错提醒
## add your code here
#include <stdio.h>
#include <string.h>

// 常量定义：最大字符串长度与最大分段数
const int MAX_STR_LEN = 100005;
const int MAX_SEG_COUNT = 15;
typedef unsigned long long HashCode;
const HashCode HASH_BASE = 131;

int file_total, seg_total;
char wildcard_type[MAX_SEG_COUNT];
char pattern_fragments[MAX_SEG_COUNT][MAX_STR_LEN];
HashCode fragment_hash[MAX_SEG_COUNT];
HashCode prefix_hash[MAX_STR_LEN];
HashCode power_table[MAX_STR_LEN];
bool dp_status[MAX_SEG_COUNT][MAX_STR_LEN];
bool star_continuable[MAX_SEG_COUNT][MAX_STR_LEN];

// 计算单个字符串的哈希值
HashCode compute_str_hash(const char *str) {
    HashCode hash_val = 0;
    for (int i = 0; str[i] != '\0'; ++i) {
        hash_val = hash_val * HASH_BASE + (unsigned char)str[i];
    }
    return hash_val;
}

// 预处理文本的前缀哈希数组，用于快速子串哈希查询
void precompute_text_hash(const char *text) {
    int text_len = strlen(text);
    prefix_hash[0] = 0;
    for (int i = 1; i <= text_len; ++i) {
        prefix_hash[i] = prefix_hash[i - 1] * HASH_BASE + (unsigned char)text[i - 1];
    }
}

// 获取文本中[left, right]区间的子串哈希值（1-based索引）
HashCode get_substring_hash(int left, int right) {
    return prefix_hash[right] - prefix_hash[left - 1] * power_table[right - left + 1];
}

// 检查文本是否与分割后的模式串匹配
bool match_check(const char *text) {
    int text_len = strlen(text);
    // 初始化DP表与标记数组
    memset(dp_status, 0, sizeof(dp_status));
    memset(star_continuable, 0, sizeof(star_continuable));
    dp_status[0][0] = true;

    // 遍历文本每个位置
    for (int pos = 0; pos < text_len; ++pos) {
        // 遍历所有分段状态
        for (int seg_idx = 0; seg_idx <= seg_total; ++seg_idx) {
            if (!dp_status[seg_idx][pos]) continue;

            // 尝试匹配下一个模式分段
            if (seg_idx < seg_total) {
                int frag_len = strlen(pattern_fragments[seg_idx + 1]);
                if (pos + frag_len <= text_len) {
                    HashCode current_sub_hash = get_substring_hash(pos + 1, pos + frag_len);
                    if (current_sub_hash == fragment_hash[seg_idx + 1]) {
                        // 根据通配符类型更新状态
                        if (wildcard_type[seg_idx + 1] == '?') {
                          
                            if (pos + frag_len < text_len) {
                                dp_status[seg_idx + 1][pos + frag_len + 1] = true;
                            }
                        } else {
                            // '*'匹配任意长度，标记可继续匹配后续字符
                            dp_status[seg_idx + 1][pos + frag_len] = true;
                            star_continuable[seg_idx + 1][pos + frag_len] = true;
                        }
                    }
                }
            }

            // 若当前状态使用了'*'，可继续匹配文本下一个字符
            if (star_continuable[seg_idx][pos] && pos + 1 <= text_len) {
                dp_status[seg_idx][pos + 1] = true;
                star_continuable[seg_idx][pos + 1] = true;
            }
        }
    }

    // 所有分段处理完成且文本匹配结束则返回true
    return dp_status[seg_total][text_len];
}

int main() {
    static char pattern[MAX_STR_LEN];
    static char filename[MAX_STR_LEN];

    // 读取模式串
    scanf("%s", pattern);
    int pattern_len = strlen(pattern);
    int seg_start = 0;
    seg_total = 0;

    // 按通配符分割模式串为多个分段
    for (int idx = 0; idx < pattern_len; ++idx) {
        if (pattern[idx] == '*' || pattern[idx] == '?') {
            int frag_len = idx - seg_start;
            seg_total++;
            memcpy(pattern_fragments[seg_total], pattern + seg_start, frag_len);
            pattern_fragments[seg_total][frag_len] = '\0';
            wildcard_type[seg_total] = pattern[idx];
            seg_start = idx + 1;
        }
    }

    // 处理最后一个无后续通配符的分段
    seg_total++;
    int last_frag_len = pattern_len - seg_start;
    memcpy(pattern_fragments[seg_total], pattern + seg_start, last_frag_len);
    pattern_fragments[seg_total][last_frag_len] = '\0';
    wildcard_type[seg_total] = '?'; // 标记为非'*'类型

    // 预计算每个分段的哈希值
    for (int i = 1; i <= seg_total; ++i) {
        fragment_hash[i] = compute_str_hash(pattern_fragments[i]);
    }

    // 预计算哈希基数的幂次，用于快速子串哈希查询
    power_table[0] = 1;
    for (int i = 1; i < MAX_STR_LEN; ++i) {
        power_table[i] = power_table[i - 1] * HASH_BASE;
    }

    // 读取文件数量
    scanf("%d", &file_total);

    // 逐个处理文件名匹配
    for (int i = 1; i <= file_total; ++i) {
        scanf("%s", filename);
        int text_len = strlen(filename);
        // 防止越界，添加一个占位字符（原代码的安全优化）
        filename[text_len] = '%';
        filename[text_len + 1] = '\0';

        precompute_text_hash(filename);
        if (match_check(filename)) {
            printf("YES\n");
        } else {
            printf("NO\n");
        }
    }

    return 0;
}

## G 汉诺塔

In [ ]:
## add your code here
import sys

def main():
    data = sys.stdin.read().split()
    if not data:
        return

    n = int(data[0])
    ops = data[1:7]

    nxt = {}

    # 记录每个柱子的最高优先级出边
    for op in ops:
        a, b = op[0], op[1]
        if a not in nxt:
            nxt[a] = b

    start = 'A'
    first = nxt[start]

    # 第三个柱子
    for ch in 'ABC':
        if ch != start and ch != first:
            third = ch
            break

    if nxt[first] == start:
        # A <-> first 形成来回型
        # 步数：2 * 3^(n-1) - 1
        ans = 2 * pow(3, n - 1) - 1
    else:
        # nxt[first] == third
        if nxt[third] == start:
            # A -> first -> third -> A 形成三元环
            # 步数：2^n - 1
            ans = pow(2, n) - 1
        else:
            # first <-> third 形成来回型
            # 步数：3^(n-1)
            ans = pow(3, n - 1)

    print(ans)

if __name__ == "__main__":
    main()

## H 马步距离

In [ ]:
## add your code here
import sys

def knight_distance(x1, y1, x2, y2):
    dx = abs(x1 - x2)
    dy = abs(y1 - y2)

    if dx < dy:
        dx, dy = dy, dx

    # 特殊情况
    if dx == 1 and dy == 0:
        return 3
    if dx == 2 and dy == 2:
        return 4

    ans = max((dx + 1) // 2, (dx + dy + 2) // 3)

    # 马每走一步，x + y 的奇偶性会改变
    if (ans + dx + dy) % 2 != 0:
        ans += 1

    return ans


def main():
    data = list(map(int, sys.stdin.buffer.read().split()))
    xp, yp, xs, ys = data
    print(knight_distance(xp, yp, xs, ys))


if __name__ == "__main__":
    main()

## I 直方图最大矩形

In [ ]:
## add your code here
from typing import List

class Solution:
    def largestRectangleArea(self, heights: List[int]) -> int:
        stack = []
        ans = 0

        # 末尾补 0，用来清空栈中剩余柱子
        heights.append(0)

        for i, h in enumerate(heights):
            while stack and heights[stack[-1]] > h:
                height = heights[stack.pop()]

                if stack:
                    width = i - stack[-1] - 1
                else:
                    width = i

                ans = max(ans, height * width)

            stack.append(i)

        heights.pop()  # 恢复原数组，避免影响外部测试
        return ans

## J 消防局的设立

In [ ]:
## add your code here
import sys

def main():
    data = list(map(int, sys.stdin.buffer.read().split()))
    if not data:
        return

    n = data[0]

    parent = [0] * (n + 1)
    depth = [0] * (n + 1)
    adj = [[] for _ in range(n + 1)]

    idx = 1
    for i in range(2, n + 1):
        p = data[idx]
        idx += 1
        parent[i] = p
        depth[i] = depth[p] + 1
        adj[i].append(p)
        adj[p].append(i)

    nodes = list(range(1, n + 1))
    nodes.sort(key=lambda x: depth[x], reverse=True)

    covered = [False] * (n + 1)

    def cover_radius_2(x):
        covered[x] = True

        for y in adj[x]:
            covered[y] = True

        for y in adj[x]:
            for z in adj[y]:
                covered[z] = True

    ans = 0

    for u in nodes:
        if covered[u]:
            continue

        # 尽量把消防局建在 u 的二级祖先处
        v = u
        if parent[v] != 0:
            v = parent[v]
        if parent[v] != 0:
            v = parent[v]

        ans += 1
        cover_radius_2(v)

    print(ans)

if __name__ == "__main__":
    main()